In [8]:
import sys
sys.path.insert(1,r'C:\Users\krajcovic\Documents\Algo\BorderSpread_tools')
import numpy as np
import pandas as pd
#from BorderSpread_tools import *
import BorderSpread.border_class as bs
import BorderSpread.capacity_class as capa
from BorderSpread.PositionManager import PositionManagerCapa, PositionManagerHedge
from BorderSpread.capacity_contract_class import CapacityContr, FwdContract

import datetime as dt
from dateutil.relativedelta import relativedelta
import calendar
import seaborn as sns
import matplotlib.pyplot as plt
import pickle

In [2]:
fut_codes = pd.read_excel(r'C:\Users\krajcovic\Documents\Trading\Data\futures codes.xlsx',
                         sheet_name='countries')
month_codes = pd.read_excel(r'C:\Users\krajcovic\Documents\Trading\Data\futures codes.xlsx',
                         sheet_name='months')

In [6]:
file_path_de = r'Z:\Data\Spot\Model\temp\de_y_2025_fcst_20240925.pkl'
file_path_fr = r'Z:\Data\Spot\Model\temp\fr_y_2025_fcst_20240925.pkl'

In [12]:
with open(file_path_de, 'rb') as f:
    de_fcst = pickle.load(f)
with open(file_path_fr, 'rb') as f:
    fr_fcst = pickle.load(f)

In [18]:
for fut1, fut2 in zip(de_fcst.mean(), fr_fcst.mean()):
    print(fut1-fut2)

12.345080989237786
-2.1768032596481532
1.3526910571251705
4.194887663951633
7.293950701861391
11.516511182033042
14.424078872886781
18.228350704487212
19.99532265465853
22.939861424824926
24.077074913392423
6.870068796937318
10.732832429526269
13.961340312042893
17.88787378012306
22.16047623363999
25.244604582359024
30.246650900910836
32.06838515908854
34.01967522495311
19.772048943034086
3.408946974020296
7.142323470717301
10.311863272044775
13.97752485156633
18.25849109719075
21.19742339003883
25.98206145016706
28.144321798016172
30.538321155564255
14.884757078860304
-0.9386263903598859
2.762336472990384
5.952165266317536
9.366165366109286
13.682879099625012
16.630444936226823
20.73231289548218
22.85726302776684
25.716025033138017


In [37]:
capa_fv_list = []
delta_list = []
spread_list = []
fut1_list = []
fut2_list = []

border_list = ['de_fr']

fut1 = 82.96
fut2 = 67.02

base_today = dt.date.today()
month_start = dt.date(base_today.year,
                      base_today.month,
                      1)



for border in border_list:
    border = [border]
    bs_class = bs.DataBorderClass(border,
                                 ['implicit'],
                                 start_date=dt.datetime(2023,9,1),
                                 end_date=dt.date.today()-dt.timedelta(days=-1,hours=1))
    #bs_class.load_data(path_spot=r'C:\Users\krajcovic\Documents\Trading\Data\Price\EEX\Spot\spot.txt')
    bs_class.load_data()
    delivery_ = ['base']
    for del_ in delivery_:
        data[del_] = bs_class.aggregate_data('M', delivery=del_)
    capacity = capa.ImplicitCapacity(border, delivery_)
    for b in border:
        for del_ in delivery_:
            capacity.capa_fit(data[del_], del_, scaling=12)
    for fut1, fut2 in zip(de_fcst.mean(), fr_fcst.mean()):        
        data = dict()
        spread = float(fut1) - float(fut2)

        delta =  capacity.capa_delta(float(fut1),
                              float(fut2),
                              'base')[0]
        capa_fv = capacity.capa_price(float(fut1),
                              float(fut2),
                              'base')[0]

        capa_fv_list.append(capa_fv)
        delta_list.append(delta)
        spread_list.append(spread)

Statement 'SELECT datetime,price FROM spot.de WHERE datetime BETWEEN '2023-09-01 00:00:00' AND '2024-09-26 23:59:00' ORDER BY datetime ASC' executed
Statement 'SELECT datetime,price FROM spot.fr WHERE datetime BETWEEN '2023-09-01 00:00:00' AND '2024-09-26 23:59:00' ORDER BY datetime ASC' executed


X:\BorderSpread\border_class.py:69: FutureWarning: The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result
  dt_vector = _data['date']['date'].dt.to_pydatetime()
X:\BorderSpread\border_class.py:77: FutureWarning: The provided callable <function mean at 0x0000028FF7AE31A0> is currently using DatetimeIndexResampler.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  _data[_border] = _data_aux.resample(freq).apply(np.mean)


In [38]:
fv_df = pd.DataFrame([border_list,
                     capa_fv_list,
                     delta_list,
                     spread_list]).T
fv_df.columns = ['border', 'fv', 'delta', 'spread']
fv_df['spread'] = fv_df['spread']*(-1)
fv_df['ext'] = np.where(fv_df['spread']>0,
                        fv_df['fv']-fv_df['spread'],
                        fv_df['fv'])

In [39]:
fv_df['ext'].mean()

6.205688002400198

In [33]:
fv_df['ext'].mean()

3.4288667800826973

In [11]:
fv_df

,border,fv,delta,spread,ext
0,de_fr,1.716067,0.351647,-3.26,1.716067
1,fr_de,4.976067,0.678574,3.26,1.716067
2,dke_de,21.330221,0.999859,21.33,0.000221
3,de_dke,0.000202,0.00017,-21.33,0.000202
4,at_de,0.081696,0.057824,-5.49,0.081696
5,de_at,5.571696,0.946157,5.49,0.081696
6,be_de,2.868191,0.536682,0.38,2.488191
7,de_be,2.488191,0.491558,-0.38,2.488191
8,be_fr,0.450062,0.221576,-2.88,0.450062
9,fr_be,3.330062,0.789951,2.88,0.450062


In [14]:
fv_df

,border,fv,delta,spread,ext
0,de_fr,1.716067,0.351647,-3.26,1.716067
1,fr_de,4.976067,0.678574,3.26,1.716067
2,dke_de,21.330221,0.999859,21.33,0.000221
3,de_dke,0.000202,0.00017,-21.33,0.000202
4,at_de,0.081696,0.057824,-5.49,0.081696
5,de_at,5.571696,0.946157,5.49,0.081696
6,be_de,2.868191,0.536682,0.38,2.488191
7,de_be,2.488191,0.491558,-0.38,2.488191
8,be_fr,0.450062,0.221576,-2.88,0.450062
9,fr_be,3.330062,0.789951,2.88,0.450062


In [31]:
fut_o1 = es('at',
           month_start.strftime('%Y-%m-%d'),
           base_today.strftime('%Y-%m-%d'))
fut_o2 = es('de',
           month_start.strftime('%Y-%m-%d'),
           base_today.strftime('%Y-%m-%d'))
    


In [32]:
fut1 = fut_o1.fwd_df_rel(fut_o1.start_date,
                  fut_o1.end_date,
                  ['M_1'], ['base'])
fut2 = fut_o2.fwd_df_rel(fut_o2.start_date,
                  fut_o2.end_date,
                  ['M_1'], ['base'])


spread = fut1 - fut2

In [33]:
spread

,M_1
2023-07-30,NaN
2023-07-31,3.25
2023-08-01,5.75
2023-08-02,5.75
2023-08-03,4.30
2023-08-04,4.25
2023-08-05,4.25
2023-08-06,4.25
2023-08-07,4.63
2023-08-08,3.75


In [50]:
capa_fv_list = []
delta_list = []
fut1_list = []
fut2_list = []
spread_list = []
border_list =['at_de', 'de_at', 'fr_de', 'de_fr']


base_today = dt.date.today()
month_start = dt.date(base_today.year,
                      base_today.month,
                      1)


for border in border_list:
    border = [border]
    bs_class = bs.DataBorderClass(border,
                                 ['implicit'],
                                 start_date=dt.datetime(2023,1,1),
                                 end_date=dt.date.today()-dt.timedelta(days=-1,hours=1))
    #bs_class.load_data(path_spot=r'C:\Users\krajcovic\Documents\Trading\Data\Price\EEX\Spot\spot.txt')
    bs_class.load_data()
    delivery_ = ['base']
    data = dict()
    for del_ in delivery_:
        data[del_] = bs_class.aggregate_data('W', delivery=del_)

    capacity = capa.ImplicitCapacity(border, delivery_)
    for b in border:
        for del_ in delivery_:
            capacity.capa_fit(data[del_], del_)
    month = 9
    year = 2023

    
    if border[0] == 'de_fr':
        fut1 = 104.25
        fut2 = 101.75
    else:
        fut2 = 104.25
        fut1 = 101.75
    spread = float(fut1) - float(fut2)
    
    delta =  capacity.capa_delta(float(fut1),
                          float(fut2),
                          'base')[0]
    capa_fv = capacity.capa_price(float(fut1),
                          float(fut2),
                          'base')[0]
    
    capa_fv_list.append(capa_fv)
    delta_list.append(delta)
    spread_list.append(spread)

In [51]:
fv_df = pd.DataFrame([border_list,
                     capa_fv_list,
                     delta_list,
                     spread_list]).T
fv_df.columns = ['border', 'fv', 'delta', 'spread']
fv_df['spread'] = fv_df['spread']*(-1)
fv_df['ext'] = np.where(fv_df['spread']>0,
                        fv_df['fv']-fv_df['spread'],
                        fv_df['fv'])

In [52]:
fv_df

,border,fv,delta,spread,ext
0,at_de,3.032175,0.759245,2.5,0.532175
1,de_at,3.032175,0.759245,2.5,0.532175
2,fr_de,4.793895,0.630975,2.5,2.293895
3,de_fr,2.293896,0.400675,-2.5,2.293896
